In [ ]:
import json
from pathlib import Path

BASE = Path("/tmp/wav2vec2")
for jsonfile in BASE.glob("*.json"):
    id = jsonfile.stem
    with open(jsonfile, "r") as f, open(BASE / f"{id}.ctm", "w") as out:
        data = json.load(f)
        for item in data["chunks"]:
            text = item["text"].lower().strip()
            start = float(item["timestamp"][0])
            end = float(item["timestamp"][1])
            out.write(f"{id} 1 {start:.2f} {end - start:.2f} {text} 1.0\n")

In [2]:
BASE = Path("/tmp/whisper")
for jsonfile in BASE.glob("*.json"):
    id = jsonfile.stem.replace(".whisper", "")

    with open(jsonfile, "r") as f, open(BASE / f"{id}.txt", "w") as out:
        data = json.load(f)
        text = data["text"].strip()
        out.write(f"{id} {text}\n")

In [ ]:
!for ctm in /tmp/wav2vec2/*.ctm; do i=$(basename $ctm .ctm); python -m sync_asr.kaldi.align_ctm_ref --ref /tmp/whisper/$i.txt --hyp /tmp/wav2vec2/$i.ctm --output /tmp/$i.ctm;done

In [11]:
!pip install num2words

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 7.1 MB/s eta 0:00:00


In [15]:
"2024"[3:]

'4'

In [19]:
num2words(2024, lang="sv")

'tvåtusentjugofyra'

In [23]:
from num2words import num2words

def basic_num_norm(a, b):
    a = a.lower().strip(",.?!;:")
    b = b.lower().strip(",.?!;:")
    if b.isdigit():
        bc = num2words(int(b), lang="sv")
        bo = num2words(int(b), lang="sv", to="ordinal")
        be = bc
        if bc.endswith("ett"):
            be = bc[:-3] + "en"
        if a == bc or a == bo or a == be:
            return True
    if len(b) == 4:
        s = b[0:2]
        e = b[2:]
        if s.isdigit() and e.isdigit():
            b = f'{num2words(int(s), lang="sv")}hundra{num2words(int(e), lang="sv")}'
            if a == b:
                return True
    if b == "§" and a in ["paragraf", "paragrafen"]:
        return True
    return False

In [56]:
BASE = Path("/tmp/pass1")
OUT = Path("/tmp/pass2")
OUT.mkdir(exist_ok=True)

OK = {
    "emtm": "MTM",
    "emteem": "MTM",
    "emtem": "MTM",
    "mtem": "MTM",
    "ntm": "MTM",
    "esokss": "SOS",
    "ämtm": "MTM",
    "ss": "S&S",
    "soss": "SOS",
    "socss": "SOS",
    "esoess": "SOS",
    "ielte": "ILT",
}

def checker(a, b):
    a = a.lower().strip(",.?!;:")
    b = b.lower().strip(",.?!;:")
    if a == b:
        return True
    if a in OK and OK[a].lower() == b:
        return True
    return False

for ctmfile in BASE.glob("*.ctm"):
    id = ctmfile.stem
    with open(ctmfile, "r") as f, open(OUT / f"{id}.ctm", "w") as out:
        for line in f:
            line = line.strip()
            parts = line.split()
            wv = parts[4]
            wh = parts[6]
            code = parts[7]
            if code == "cor":
                out.write(line + "\n")
            elif code == "sub":
                if checker(wv, wh):
                    if parts[6] == "SOS":
                        parts[6] = "S&S"
                    parts[7] = "cor"
                    parts[4] = parts[6]
                    out.write(" ".join(parts) + "\n")
                elif basic_num_norm(wv, wh):
                    parts[7] = "norm"
                    out.write(" ".join(parts) + "\n")
                else:
                    out.write(line + "\n")
            else:
                out.write(line + "\n")

In [57]:
def move_ins(line1, line2):
    parts1 = line1.split()
    parts2 = line2.split()

    if parts1[7] == "ins" and parts1[6] == "-":
        a = parts1[4]
        b = parts2[6]
        bnorm = b.strip(",.?!;:").lower()
        if a == bnorm:
            parts1[6] = b
            parts1[7] = "cor"
            parts2[7] = "ins"
            parts2[6] = "-"
            return [" ".join(parts1), " ".join(parts2)]
    elif parts2[7] == "ins" and parts2[6] == "-":
        a = parts2[4]
        b = parts1[6]
        bnorm = b.strip(",.?!;:").lower()
        if a == bnorm:
            parts2[6] = b
            parts2[7] = "cor"
            parts1[7] = "ins"
            parts1[6] = "-"
            return [" ".join(parts1), " ".join(parts2)]
    return [line1, line2]

def merge_ins(line1, line2):
    parts1 = line1.split()
    parts2 = line2.split()

    if parts1[7] == "ins" or parts2[7] == "ins":
        merge_a = parts1[4] + parts2[4]
        b = parts1[6] if parts2[7] == "ins" else parts2[6]
        bnorm = b.strip(",.?!;:").lower()
        if checker(merge_a, b):
            if bnorm == "sos":
                b = b.replace("SOS", "S&S")
            output = [parts1[0], parts1[1], parts1[2]]
            b_end = float(parts2[2]) + float(parts2[3])
            new_dur = b_end - float(parts1[2])
            output.append(f"{new_dur:.2f}")
            output.append(b)
            output.append("1.0")
            output.append(b)
            output.append("cor")
            return [" ".join(output)]
    return [line1, line2]

In [41]:
move_ins("CA37298 1 31.86 0.38 medier 1.0 - ins", "CA37298 1 32.74 0.02 m 1.0 medier, sub")

['CA37298 1 31.86 0.38 medier 1.0 medier, cor',
 'CA37298 1 32.74 0.02 m 1.0 - ins']

In [45]:
merge_ins("CA37298 1 32.74 0.02 m 1.0 - ins", "CA37298 1 32.96 0.52 tm 1.0 MTM, sub")

ins sub
mtm MTM, mtm


['CA37298 1 32.74 0.74 mtm 1.0 MTM, cor']

In [58]:
BASE = Path("/tmp/pass3")
OUT = Path("/tmp/pass4")
OUT.mkdir(exist_ok=True)

# move ins
for ctmfile in BASE.glob("*.ctm"):
    id = ctmfile.stem
    with open(ctmfile, "r") as f, open(OUT / f"{id}.ctm", "w") as out:
        lines = [line.strip() for line in f]
        i = 0
        while i < len(lines):
            if i < len(lines) - 1:
                newlines = move_ins(lines[i], lines[i + 1])
                out.write(newlines[0] + "\n")
                if len(newlines) == 2:
                    out.write(newlines[1] + "\n")
                    i += 2
                else:
                    i += 1
            else:
                out.write(lines[i] + "\n")
                i += 1

In [59]:
BASE = Path("/tmp/pass4")
OUT = Path("/tmp/pass5")
OUT.mkdir(exist_ok=True)

for ctmfile in BASE.glob("*.ctm"):
    id = ctmfile.stem
    with open(ctmfile, "r") as f, open(OUT / f"{id}.ctm", "w") as out:
        lines = [line.strip() for line in f]
        # slide a 2 window over lines
        i = 0
        while i < len(lines) - 1:
            line1 = lines[i]
            line2 = lines[i + 1]
            new_lines = merge_ins(line1, line2)
            if len(new_lines) == 1:
                out.write(new_lines[0] + "\n")
                i += 2
            else:
                out.write(new_lines[0] + "\n")
                i += 1
        if i == len(lines) - 1:
            out.write(lines[i] + "\n")